In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import os
import FMfuncs
import configs
import plot_func

import PGFMv2 as PGFM


os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"


In [ ]:

PGFMclass = PGFM.PGFM()

In [ ]:
if configs.dataset == "two_uniform_box":
    dataset = np.load(r'./data/uniform2.npy') #cropped_Gaussian uniform uniform2
elif configs.dataset == "uniform_box":
    dataset = np.load(r'./data/uniform.npy')
elif configs.dataset == "cropped_Gaussian":
    dataset = np.load(r'./data/cropped_Gaussian.npy')
else:
    raise ValueError

dataset = torch.tensor(dataset,dtype=torch.float32, device=configs.device)
x_mat = PGFMclass.get_samples(dataset, 10000)
plot_func.plot_scatter_with_info(x_mat.cpu().numpy().transpose(), 0, 'unconstrained2', fix_bound = False,numItermax = 200000, distance = False)



In [ ]:
if configs.dataset == "two_uniform_box":
    path = './saved_model/FM_2uniform_1000000.pth' #cropped_Gaussian uniform uniform2
elif configs.dataset == "uniform_box":
    path = './saved_model/FM_uniform2_1000000.pth'
elif configs.dataset == "cropped_Gaussian":
    path = './saved_model/FM_test2_1000000.pth'
else:
    raise ValueError

# trained_model = PGFMclass.train2_2stage(dataset, path)
trained_model = PGFMclass.train2_2stage_distance(dataset, path)
#FM_test2_1000000: CG
#FM_2uniform_1000000: 2uniform
#FM_uniform2_1000000: uniform

In [ ]:
if configs.dataset == "two_uniform_box":
    ckpt1 = torch.load('./saved_model/FM_2uniform_1000000.pth', map_location=configs.device, weights_only= True)
    # ckpt2 = torch.load('./saved_model/PGFM_2uniform_20000.pth', map_location=configs.device, weights_only= True)
    ckpt2 = torch.load('./saved_model/distanceFM_2uniform_5000.pth', map_location=configs.device, weights_only= True)
    #PGFM_2uniform_Mar27
    #PGFM_2uniform_only_terminal
elif configs.dataset == "uniform_box":
    ckpt1 = torch.load('./saved_model/FM_uniform2_1000000.pth', map_location=configs.device, weights_only= True)
    ckpt2 = torch.load('./saved_model/PGFM_uniform_20000.pth', map_location=configs.device, weights_only= True)
elif configs.dataset == "cropped_Gaussian":
    ckpt1 = torch.load('./saved_model/FM_test2_1000000.pth', map_location=configs.device, weights_only= True)
    # ckpt2 = torch.load('./saved_model/PGFM_CG_20000.pth', map_location=configs.device, weights_only= True)
    ckpt2 = torch.load('./saved_model/distanceFM_CG_5000.pth', map_location=configs.device, weights_only= True)
    # PGFM_CG_Mar27
else:
    raise ValueError


In [ ]:

stage1model = PGFMclass.get_untrained_model()
stage1model.load_state_dict(ckpt1)

PGFMclass.policy.load_state_dict(ckpt2)
stage2model = PGFMclass.policy

res = PGFMclass.PGFMsample(stage1model, stage2model, 10000)



reference = PGFMclass.get_samples(dataset, 10000).cpu().numpy()
plot_func.plot_scatter_with_info(res.cpu().numpy().transpose(), reference, 'Mar27_2uniform_CG', fix_bound = False,numItermax = 200000, distance = True)




In [ ]:
res = PGFMclass.PGFMsample(stage1model, stage2model, 50000).cpu().numpy()
# res = PGFMclass.get_samples(dataset, 50000).cpu().numpy()
# res = FMfuncs.sampler(stage1model, 50000, stoptime=1).cpu().numpy()
xy = res
outliner_mat = np.logical_or(np.abs(np.abs(xy[:, 0]) - configs.uniform_center) >= configs.bound,
                             np.abs(np.abs(xy[:, 1]) - configs.uniform_center) >= configs.bound)
xy_out = xy[outliner_mat]

square = plt.Polygon(
    [[- configs.bound + configs.uniform_center, - configs.bound + configs.uniform_center],
     [configs.bound + configs.uniform_center, - configs.bound + configs.uniform_center],
     [configs.bound + configs.uniform_center, configs.bound + configs.uniform_center],
     [- configs.bound + configs.uniform_center, configs.bound + configs.uniform_center]],
    closed=True, edgecolor='black', fill=False, linewidth=1.2
)

square1 = plt.Polygon(
    [[- configs.bound - configs.uniform_center, - configs.bound - configs.uniform_center],
     [configs.bound - configs.uniform_center, - configs.bound - configs.uniform_center],
     [configs.bound - configs.uniform_center, configs.bound - configs.uniform_center],
     [- configs.bound - configs.uniform_center, configs.bound - configs.uniform_center]],
    closed=True, edgecolor='black', fill=False, linewidth=1.2
)
fig, ax = plt.subplots()
H, xedges, yedges = np.histogram2d(xy[:, 0], xy[:, 1], bins=100)

H_transformed = H**0.2  # sqrt intensifies visual difference (log-like)
H_masked = np.ma.masked_where(H == 0, H_transformed)
c = ax.pcolormesh(xedges, yedges, H_masked.T, cmap='Blues')
ax.add_patch(square)
ax.add_patch(square1)
ax.set_aspect('equal')
ax.scatter(xy_out[:, 0], xy_out[:, 1], s=15, alpha=1, color='red')
# plt.colorbar(scatter, label='Density')

# Set axis limits to clearly show the 5x5 area
if True == True:
    plt.xlim(-configs.bound * 1.2 - configs.uniform_center, configs.bound * 1.2 + configs.uniform_center)
    plt.ylim(-configs.bound * 1.2 - configs.uniform_center, configs.bound * 1.2 + configs.uniform_center)

# plt.xlabel('$x_1$')
# plt.ylabel('$x_2$')
# ax.axis('off')
ax.tick_params(labelsize=30)
ax.set_xticks([-5, 0, 5])
ax.set_yticks([-5, 0, 5])
plt.savefig('./fig/' + '2uniform_distanceFM' + '.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:


FMnum_out_record = np.array([])
FMprob_record = np.array([])
FMSWD_record = np.array([])
PGFMnum_out_record = np.array([])
PGFMprob_record = np.array([])
PGFMSWD_record = np.array([])

for i in range(100):
    if i%10 == 0:
        print(i)
    reference = PGFMclass.get_samples(dataset, 10000).cpu().numpy()
    resFM = FMfuncs.sampler(stage1model, 10000, stoptime=1)
    resPGFM = PGFMclass.PGFMsample(stage1model, stage2model, 10000)
    
    FMnum_out, FMprob, FMSWD = plot_func.only_info(resFM.cpu().numpy().transpose(), reference)
    PGFMnum_out, PGFMprob, PGFMSWD = plot_func.only_info(resPGFM.cpu().numpy().transpose(), reference)
    
    FMnum_out_record = np.append(FMnum_out_record, FMnum_out)
    FMprob_record = np.append(FMprob_record, FMprob)
    FMSWD_record = np.append(FMSWD_record, FMSWD)
    PGFMnum_out_record = np.append(PGFMnum_out_record, PGFMnum_out)
    PGFMprob_record = np.append(PGFMprob_record, PGFMprob)
    PGFMSWD_record = np.append(PGFMSWD_record, PGFMSWD)
    
#     print(FMnum_out, PGFMnum_out)

# np.savez( './result_record/' +'uniform2_record.npz', FMnum_out_record=FMnum_out_record, FMprob_record=FMprob_record,
#         FMSWD_record=FMSWD_record,
#         PGFMnum_out_record=PGFMnum_out_record,
#         PGFMprob_record=PGFMprob_record,
#         PGFMSWD_record=PGFMSWD_record)


In [ ]:
print(" FM SWD: ", f"{np.mean( FMSWD_record):.4f}", r"\pm", f"{np.std( FMSWD_record):.4f}")
print(" FM num out: ", f"{np.mean( FMnum_out_record):.4f}", r"\pm", f"{np.std( FMnum_out_record):.4f}")

print("PGFM SWD: ", f"{np.mean(PGFMSWD_record):.4f}", r"\pm", f"{np.std(PGFMSWD_record):.4f}")
print("PGFM num out: ", f"{np.mean(PGFMnum_out_record):.4f}", r"\pm", f"{np.std(PGFMnum_out_record):.4f}")

In [ ]:
FMclass = FMfuncs.OTFlowMatching()

In [ ]:
trained_model = FMclass.train(dataset)